# 02 · Fine-tuning a Transformer for Text
### Whose language, whose labels, whose values?

Language technology can **support** communities — transcription, translation,
search, and helping revitalise under-resourced languages. But language models
learn from text written by *some* people, and they are fine-tuned on labels
chosen by *some* people. Those choices get baked into the model.

In this notebook you will **fine-tune** a small pretrained transformer to
classify the **emotion** of short messages. Fine-tuning means: take a model that
already understands general language, then train it a little more on your specific
task. It is the single most common way communities and small teams adapt AI.

Emotion is a deliberately tricky choice, because **how emotion is expressed and
categorised is deeply cultural**. That gives us a lot to reflect on:

> **Whose emotional expression counts as "standard"? Who labelled this data, and
> would your community draw the categories the same way?**


> ✅ **Solution / runnable reference version.** This is the fully-solved version
> of the workshop notebook: every *Your turn* blank is filled in and code comments
> are trimmed, so you can **Runtime → Run all** and confirm it works end to end.
> The teaching version (with blanks to complete) lives in the parent `workshop3/` folder.


## How to use this notebook

This is **Workshop 3** of the *Introduction to AI* series. Workshops 1 and 2
looked at tabular data, images, tokenization and retrieval. Workshop 3 is a
hands-on tour of the **Hugging Face** ecosystem across three applications.

The notebooks are designed to be run in **Google Colab**, top to bottom.

Each notebook is organised in three levels:

- 🟢 **Beginner** — run pretrained models with a few lines of code.
- 🟡 **Medium** — adapt models, change parameters, and read the results critically.
- 🔴 **Optional / Advanced** — train or fine-tune a model yourself (best with a GPU).

Look out for these blocks:

- `# TODO:` cells are **your turn to code**. Try them before scrolling to the solution.
- **▶️ Play with it** cells are safe to edit and rerun — change models, labels, or numbers.
- **🧭 Critical reflection** boxes connect the technical work to the bigger questions of
the workshop: *who designs these systems, who do they serve, whose knowledge do they
include, and whose values do they encode?*

> 💡 **Before you start:** switch on a GPU via
> `Runtime -> Change runtime type -> Hardware accelerator -> GPU`.
> Everything still runs on CPU, just more slowly.

Workshop 3 sequence:
1. `01_computer_vision_with_transformers.ipynb`
2. `02_finetuning_transformers_for_text.ipynb`
3. `03_transformers_for_sensor_time_series.ipynb`


### 🧭 The idea of fine-tuning in one picture

```
Pretrained model            +   Your labelled data      =   Fine-tuned model
(knows general language)        (your task & values)        (does your task)
```

- **Pretraining** is expensive and done once by large organisations on huge text.
- **Fine-tuning** is cheap and is where *your* data and *your* labels shape the model.
- This is exactly where community values, consent, and bias enter the system.


In [ ]:
import random
import sys
from pathlib import Path

import numpy as np

try:
    import torch
except ImportError:
    torch = None

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

if torch is not None:
    torch.manual_seed(SEED)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(SEED)

DEVICE = "cuda" if (torch is not None and torch.cuda.is_available()) else "cpu"

PIPELINE_DEVICE = 0 if DEVICE == "cuda" else -1

print("Python version:", sys.version.split()[0])
print("Detected device:", DEVICE)
if DEVICE == "cpu":
    print("Tip: enable a GPU via Runtime -> Change runtime type for faster runs.")
print("Seed set to:", SEED)

NOTEBOOK_FRAMING = "fine-tuning language models, with attention to representation and bias"
NOTEBOOK_FRAMING


In [ ]:
!pip -q install transformers datasets scikit-learn

print("Fine-tuning notebook ready once transformers, datasets and scikit-learn are available.")


## 🟢 Part 1 — Load and inspect the data

We use the public **emotion** dataset: short English messages, each labelled with
one of six emotions. Always *look at your data* before training — the data is the
model's entire view of the world.


In [ ]:
from datasets import load_dataset

dataset = load_dataset("dair-ai/emotion")
print(dataset)

label_names = dataset["train"].features["label"].names
print("Emotion labels:", label_names)


In [ ]:
import pandas as pd

sample = dataset["train"].select(range(8))
for row in sample:
    print(f"[{label_names[row['label']]:<8}] {row['text']}")


In [ ]:
counts = pd.Series([label_names[l] for l in dataset["train"]["label"]]).value_counts()
print(counts)
counts.plot(kind="bar", title="Examples per emotion (train split)")


### 🧭 Critical reflection: read the labels first

- These six categories come from one research team's taxonomy. Many cultures name,
  blend, or value emotions differently — some important feelings simply have no box here.
- The text is English social-media style. A model trained here may misread other
  dialects, registers, or languages.
- Notice the class imbalance. A model can score well overall while being poor on
  the *rare* classes — which are often the ones that matter most.

**Discuss:** if your community were labelling this data, which categories would you
add, merge, or remove? Who should get to decide?


## 🟡 Part 2 — Tokenize the text

Models do not read characters; they read **token ids**. The tokenizer must match
the model. We will fine-tune `distilbert-base-uncased` (a small, fast BERT), so we
load its matching tokenizer.


In [ ]:
from transformers import AutoTokenizer

model_checkpoint = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

demo = tokenizer("I can't believe how good this feels today!")
print(demo)
print("Tokens:", tokenizer.convert_ids_to_tokens(demo["input_ids"]))


In [ ]:
def tokenize_function(batch):
    return tokenizer(batch["text"], truncation=True)

tokenized = dataset.map(tokenize_function, batched=True)
print(tokenized["train"][0].keys())


In [ ]:
TRAIN_SIZE = 2000
EVAL_SIZE = 500

small_train = tokenized["train"].shuffle(seed=SEED).select(range(TRAIN_SIZE))
small_eval = tokenized["validation"].shuffle(seed=SEED).select(range(EVAL_SIZE))
print("Using", len(small_train), "train and", len(small_eval), "eval examples.")


## 🟡 Part 3 — Load the model and define metrics

We load DistilBERT with a fresh classification head sized to our 6 labels, and we
pass `id2label` / `label2id` so predictions come back as readable names.


In [ ]:
from transformers import AutoModelForSequenceClassification

id2label = {i: name for i, name in enumerate(label_names)}
label2id = {name: i for i, name in enumerate(label_names)}

model = AutoModelForSequenceClassification.from_pretrained(
    model_checkpoint,
    num_labels=len(label_names),
    id2label=id2label,
    label2id=label2id,
)


In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score, f1_score

def compute_metrics(eval_pred):
    logits, gold = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(gold, preds),

        "macro_f1": f1_score(gold, preds, average="macro"),
    }


In [ ]:
from transformers import TrainingArguments, Trainer, DataCollatorWithPadding

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

def build_training_args(**kwargs):

    try:
        return TrainingArguments(eval_strategy="epoch", **kwargs)
    except TypeError:
        return TrainingArguments(evaluation_strategy="epoch", **kwargs)

training_args = build_training_args(
    output_dir="distilbert-emotion",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    save_strategy="no",
    logging_steps=50,
    report_to="none",
)


In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=small_train,
    eval_dataset=small_eval,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

trainer.train()
metrics = trainer.evaluate()
print(metrics)


In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

preds_output = trainer.predict(small_eval)
y_pred = np.argmax(preds_output.predictions, axis=-1)
y_true = preds_output.label_ids

cm = confusion_matrix(y_true, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=label_names)
fig, ax = plt.subplots(figsize=(7, 7))
disp.plot(ax=ax, xticks_rotation=45)
plt.title("Which emotions does the model confuse?")
plt.show()


### ▶️ Play with it — test your own sentences

Edit the sentences below. Deliberately try **dialects, slang, code-switching, and
non-Western phrasing**. Where does the model get confused, and why might that be?


In [ ]:
import torch

def predict_emotion(text):
    inputs = tokenizer(text, return_tensors="pt", truncation=True).to(model.device)
    with torch.no_grad():
        logits = model(**inputs).logits
    probs = logits.softmax(-1).squeeze()
    top = int(probs.argmax())
    print(f"'{text}'")
    print(f"   -> {label_names[top]}  (confidence {probs[top]:.2f})")
    return label_names[top]

for s in [
    "I am so happy to see my family again",
    "wallah I'm shaking I'm so nervous",
    "that's sick, I can't wait!!",
    "I'm not mad, just disappointed",
]:
    predict_emotion(s)


### 🧭 Critical reflection: bias is learned, not invented by the model

- The model only knows the patterns in its training text. Slang, dialects, and
  other languages that were rare in the data are predicted less reliably.
- "Confidence" is just a number from a softmax. A wrong answer can look very confident.
- If this fed a real moderation or wellbeing system, *whose* messages would be
  misread most often — and what would the consequences be for them?


## 🧪 More experiments — a reusable "knob-box"

Good experiments change **one variable at a time**. The helper below retrains a
fresh model with whatever settings you pass, then returns its scores — so you can
sweep data size, learning rate, or freezing without rewriting the training code.


In [ ]:
def run_experiment(train_size=1000, lr=2e-5, epochs=2, freeze_base=False, ckpt=model_checkpoint, seed=SEED):

    m = AutoModelForSequenceClassification.from_pretrained(
        ckpt, num_labels=len(label_names), id2label=id2label, label2id=label2id,
    )
    if freeze_base:

        for p in m.base_model.parameters():
            p.requires_grad = False

    train_subset = tokenized["train"].shuffle(seed=seed).select(range(train_size))
    args = build_training_args(
        output_dir="exp",
        learning_rate=lr,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=32,
        num_train_epochs=epochs,
        save_strategy="no",
        logging_strategy="no",
        disable_tqdm=True,
        report_to="none",
    )
    trainer = Trainer(
        model=m, args=args, train_dataset=train_subset, eval_dataset=small_eval,
        data_collator=data_collator, compute_metrics=compute_metrics,
    )
    trainer.train()
    scores = trainer.evaluate()
    return {"train_size": train_size, "lr": lr, "epochs": epochs, "freeze_base": freeze_base,
            "accuracy": scores["eval_accuracy"], "macro_f1": scores["eval_macro_f1"]}

print(run_experiment(train_size=500, epochs=1))


### 🧪 Experiment A — how much data do you actually need?

**Why this experiment?** Labelled data is the scarcest resource for most community
projects. Training on growing slices and plotting the curve shows where accuracy
*stops* improving — i.e. how much labelling effort is genuinely worth it, and how
much a low-resource language or community is disadvantaged by having less data.


In [ ]:
import matplotlib.pyplot as plt

sizes = [100, 250, 500, 1000, 2000]
curve = [run_experiment(train_size=s, epochs=1) for s in sizes]

plt.figure(figsize=(7, 4))
plt.plot(sizes, [c["accuracy"] for c in curve], marker="o", label="accuracy")
plt.plot(sizes, [c["macro_f1"] for c in curve], marker="s", label="macro-F1")
plt.xlabel("number of training examples")
plt.ylabel("score on validation set")
plt.title("Data-scaling curve: more labels -> diminishing returns")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()


### 🧪 Experiment B — feature extraction vs. full fine-tuning

**Why this experiment?** "Fine-tuning" can mean two very different things. If we
**freeze** the transformer body and train only the small classification head, we use
far less compute and memory — but can the frozen features already solve the task?
Comparing the two reveals *what fine-tuning is actually buying you*.


In [ ]:
frozen = run_experiment(train_size=2000, epochs=3, freeze_base=True)
full = run_experiment(train_size=2000, epochs=3, freeze_base=False)

print("Frozen body (head only):", frozen)
print("Full fine-tune:         ", full)
print()
print("Frozen trains far fewer parameters and is faster. Was the accuracy worth the extra cost?")


### ▶️ Play with it — learning-rate sweep

**Why this experiment?** Learning rate is the single most important knob. Too high
and training is unstable; too low and it barely learns. Sweeping it builds intuition
you will reuse on *every* future model.


In [ ]:
lrs = [1e-5, 2e-5, 5e-5, 1e-4]
lr_results = [run_experiment(train_size=1000, lr=lr, epochs=2) for lr in lrs]

for r in lr_results:
    print(f"lr={r['lr']:.0e}  accuracy={r['accuracy']:.3f}  macro_f1={r['macro_f1']:.3f}")


### 🧪 Experiment C — do you even need to fine-tune? (zero-shot baseline)

**Why this experiment?** Before spending effort fine-tuning, check what a general
**zero-shot** classifier already does: you just hand it the label names. If zero-shot
is good enough, you may not need to train at all; if it is weak, you have *measured*
the value your fine-tuning added. (The model below is a large download.)


In [ ]:
import torch
from transformers import pipeline

zero_shot_text = pipeline("zero-shot-classification", model="facebook/bart-large-mnli", device=PIPELINE_DEVICE)

probe = [
    "I am so happy to see my family again",
    "I'm terrified of what comes next",
    "why would you do that to me",
]
for text in probe:

    out = zero_shot_text(text, candidate_labels=label_names)
    zs_label = out["labels"][0]

    inputs = tokenizer(text, return_tensors="pt", truncation=True).to(model.device)
    with torch.no_grad():
        ft_label = label_names[int(model(**inputs).logits.argmax(-1))]
    print(f"'{text}'")
    print(f"   zero-shot  -> {zs_label} ({out['scores'][0]:.2f})")
    print(f"   fine-tuned -> {ft_label}")


## 🔴 Optional / Advanced — efficient fine-tuning with LoRA

Full fine-tuning updates **every** weight in the model. **LoRA** (Low-Rank
Adaptation) freezes the original model and trains a tiny number of extra
parameters instead. It is faster, uses far less memory, and produces small
shareable "adapters" — which matters a lot for communities without big GPUs.


In [ ]:
!pip -q install peft

from peft import LoraConfig, get_peft_model, TaskType

base_model = AutoModelForSequenceClassification.from_pretrained(
    model_checkpoint,
    num_labels=len(label_names),
    id2label=id2label,
    label2id=label2id,
)


In [ ]:
lora_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=8,
    lora_alpha=16,
    lora_dropout=0.1,
    target_modules=["q_lin", "v_lin"],
)

lora_model = get_peft_model(base_model, lora_config)
lora_model.print_trainable_parameters()


In [ ]:
lora_args = build_training_args(
    output_dir="distilbert-emotion-lora",
    learning_rate=1e-3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    save_strategy="no",
    logging_steps=50,
    report_to="none",
)

lora_trainer = Trainer(
    model=lora_model,
    args=lora_args,
    train_dataset=small_train,
    eval_dataset=small_eval,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

lora_trainer.train()
print("LoRA results:", lora_trainer.evaluate())


## 🧭 Closing reflection & exercises

**Reflection**
- LoRA trained a tiny fraction of the parameters. Compare its accuracy/F1 to the
  full fine-tune. Was the trade-off worth it? When would each be the right choice?
- Cheaper fine-tuning means *more* people can adapt models — including communities.
  It also means more people can build harmful systems. Tools are dual-use.

**Exercises**
1. Sweep the learning rate (`1e-5`, `2e-5`, `5e-5`). Plot accuracy vs. learning rate.
2. Swap `distilbert-base-uncased` for a **multilingual** model
   (`distilbert-base-multilingual-cased`) and test non-English sentences.
3. Increase `TRAIN_SIZE`. How much data do you need before results stop improving?
4. Write 5 sentences in a dialect or language you know well. Is the model fair to
   them? Document what you find — this is exactly how real audits begin.
